# Edu Nexus – Phase 1: Hybrid PDF to DOCX Converter

This notebook implements a hybrid PDF conversion strategy.
It first attempts local text extraction.
If the extracted text is insufficient, the PDF is marked for OCR fallback.

This aligns with the "Local First, OCR Fallback" rule.


In [9]:
! pip install pymupdf

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/18.4 MB ? eta -:--:--
   --------------- ------------------------ 7.1/18.4 MB 43.7 MB/s eta 0:00:01
   ---------------------- ----------------- 10.2/18.4 MB 42.6 MB/s eta 0:00:01
   ---------------------------- ----------- 13.1/18.4 MB 22.2 MB/s eta 0:00:01
   -------------------------------- ------- 14.9/18.4 MB 18.8 MB/s eta 0:00:01
   ----------------------------------- ---- 16.3/18.4 MB 16.3 MB/s eta 0:00:01
   ------------------------------------- -- 17.3/18.4 MB 14.5 MB/s eta 0:00:01
   ---------------------------------------- 18.4/18.4 MB 12.9 MB/s eta 0:00:00



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import fitz  # PyMuPDF
from pathlib import Path
from docx import Document
from docx.shared import Pt


In [11]:


def extract_text_from_pdf(pdf_path: Path) -> str:
    """
    Attempts to extract selectable text from a PDF using PyMuPDF.
    """
    doc = fitz.open(pdf_path)
    text_chunks = []

    for page in doc:
        page_text = page.get_text().strip()
        if page_text:
            text_chunks.append(page_text)

    return "\n".join(text_chunks)


In [12]:
# ===== HYBRID PDF CLASSIFIER =====

def is_scanned_pdf(extracted_text: str, threshold: int = 50) -> bool:
    """
    Determines if a PDF is likely scanned based on text length.
    """
    return len(extracted_text.strip()) < threshold


In [13]:
# ===== PDF TO DOCX (LOCAL ONLY) =====

def convert_pdf_to_docx(pdf_path: Path, output_path: Path):
    """
    Converts a PDF to DOCX using local extraction.
    Falls back to OCR in later stages if needed.
    """
    extracted_text = extract_text_from_pdf(pdf_path)

    if is_scanned_pdf(extracted_text):
        return {
            "status": "ocr_required",
            "file": pdf_path.name
        }

    doc = Document()
    doc.add_heading(pdf_path.stem, level=1)

    for line in extracted_text.split("\n"):
        para = doc.add_paragraph(line)
        for run in para.runs:
            run.font.size = Pt(11)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    doc.save(output_path)

    return {
        "status": "converted",
        "output": str(output_path)
    }


In [14]:
# ===== TEST PDF CONVERSION =====

pdf_input = Path("../../edu_nexus_db/raw/pdf")
pdf_files = list(pdf_input.glob("*.pdf"))

if not pdf_files:
    raise FileNotFoundError("No PDF files found in raw/pdf folder.")

results = []

for pdf_file in pdf_files:
    output_docx = Path("../../edu_nexus_db/normalized/docx") / f"{pdf_file.stem}.docx"
    result = convert_pdf_to_docx(pdf_file, output_docx)
    results.append((pdf_file.name, result))

results


[('Classes in Java.pdf',
  {'status': 'converted',
   'output': '..\\..\\edu_nexus_db\\normalized\\docx\\Classes in Java.docx'})]